# MODIS-Only Download from GEE

Exports MODIS 8-day composites for Indonesia (land only, ≤126°E) to Google Drive.

**Change `YEARS` below** to download a different year or range.

In [1]:
# Setup: GEE auth
!pip install -q earthengine-api
import ee
from google.colab import auth
auth.authenticate_user()
ee.Initialize(project="flooding-process")
print("✓ GEE ready")

✓ GEE ready


In [2]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive mounted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Drive mounted


In [3]:
# CONFIG: Change year(s) here
YEARS = range(2017,2022)  # e.g. [2021] or [2019, 2020, 2021] or range(2017, 2022)
GEE_FOLDER = 'indonesia-flood-cnn-lstm/data/modis_8day_5d'  # Drive folder
MAX_QUEUED = 2500
BATCH_WAIT = 120

In [4]:
# Indonesia land only, clipped to 126°E
countries = ee.FeatureCollection('FAO/GAUL/2015/level0')
indonesia_land = countries.filter(ee.Filter.eq('ADM0_NAME', 'Indonesia')).geometry()
bbox = ee.Geometry.Rectangle([95.0, -11.0, 126.0, 6.0])
indonesia_region = indonesia_land.intersection(bbox)
grid_size = 5.0

In [5]:
def get_grid_cells(roi, grid_size):
    bounds = roi.bounds().getInfo()['coordinates'][0]
    min_x, max_x = min(p[0] for p in bounds), max(p[0] for p in bounds)
    min_y, max_y = min(p[1] for p in bounds), max(p[1] for p in bounds)
    features = []
    x = min_x
    while x < max_x:
        y = min_y
        while y < max_y:
            geom = ee.Geometry.Rectangle([x, y, min(x+grid_size, max_x), min(y+grid_size, max_y)])
            features.append(ee.Feature(geom))
            y += grid_size
        x += grid_size
    fc = ee.FeatureCollection(features)
    filtered = fc.filterBounds(roi)
    info = filtered.getInfo()
    return [ee.Geometry(f['geometry']) for f in info.get('features', [])]

In [6]:
from datetime import datetime, timedelta

def get_8day_periods(year):
    d = datetime(year, 1, 1)
    periods = []
    while d.year == year:
        end = min(d + timedelta(days=7), datetime(year, 12, 31))
        periods.append((d.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d')))
        d += timedelta(days=8)
    return periods

land_mask = ee.Image("MODIS/006/MOD44W/2015_01_01").select('water_mask').eq(1)

In [7]:
# Tiling
tiles = get_grid_cells(indonesia_region, grid_size)
n_periods = sum(len(get_8day_periods(y)) for y in YEARS)
total = len(tiles) * n_periods
print(f"Tiles: {len(tiles)} | Periods: {n_periods} | Total exports: {total:,}")

Tiles: 23 | Periods: 230 | Total exports: 5,290


In [8]:
# Export loop (resume-safe: skips already queued/completed)
import time

def count_queued():
    return sum(1 for t in ee.batch.Task.list() if t.state in ('RUNNING', 'READY'))

def already_done():
    return {t.config.get('description', '') for t in ee.batch.Task.list()
            if t.state in ('RUNNING', 'READY', 'COMPLETED')}

done = already_done()
print(f"Skipping {len(done)} already queued/completed")
submitted, skipped = 0, 0

for year in YEARS:
    for p_idx, (start_d, end_d) in enumerate(get_8day_periods(year)):
        for i, tile_roi in enumerate(tiles):
            name = f'Indo_5d_MODIS_{year}_{p_idx:03d}_Tile_{i}'
            if name in done:
                skipped += 1
                continue
            while count_queued() >= MAX_QUEUED:
                print(f"  Waiting ({count_queued()} queued)...")
                time.sleep(BATCH_WAIT)
            modis = ee.ImageCollection("MODIS/061/MOD09A1").filterBounds(tile_roi).filterDate(start_d, end_d)\
                .select(['sur_refl_b01','sur_refl_b02','sur_refl_b03','sur_refl_b04','sur_refl_b05','sur_refl_b06','sur_refl_b07'])
            modis = modis.map(lambda img: img.updateMask(land_mask)).mosaic()
            task = ee.batch.Export.image.toDrive(image=modis, description=name, folder=GEE_FOLDER,
                scale=500, region=tile_roi, crs='EPSG:4326', maxPixels=1e10, fileFormat='GeoTIFF')
            task.start()
            submitted += 1
            done.add(name)
        print(f"  {year} period {p_idx} done")

print(f"Total: submitted={submitted} skipped={skipped}")

Skipping 16751 already queued/completed
  2017 period 0 done
  2017 period 1 done
  2017 period 2 done
  2017 period 3 done
  2017 period 4 done
  2017 period 5 done
  2017 period 6 done
  2017 period 7 done
  2017 period 8 done
  2017 period 9 done
  2017 period 10 done
  2017 period 11 done
  2017 period 12 done
  2017 period 13 done
  2017 period 14 done
  2017 period 15 done
  2017 period 16 done
  2017 period 17 done
  2017 period 18 done
  2017 period 19 done
  2017 period 20 done
  2017 period 21 done
  2017 period 22 done
  2017 period 23 done
  2017 period 24 done
  2017 period 25 done
  2017 period 26 done
  2017 period 27 done
  2017 period 28 done
  2017 period 29 done
  2017 period 30 done
  2017 period 31 done
  2017 period 32 done
  2017 period 33 done
  2017 period 34 done
  2017 period 35 done
  2017 period 36 done
  2017 period 37 done
  2017 period 38 done
  2017 period 39 done
  2017 period 40 done
  2017 period 41 done
  2017 period 42 done
  2017 period 43 done
  

  2021 period 43 done
  2021 period 44 done
  2021 period 45 done
Total: submitted=72 skipped=5218
